# Machine Learning Fundamentals: Hands-on Implementation

> **"Connecting theoretical principles to a practical Python workflow."**

Welcome to the practical lab notebook for **Introduction to Machine Learning**. In this notebook, we will implement a complete, production-ready machine learning classification pipeline. We will build intuition, preprocess data, train a linear model, evaluate metrics, and experiment with hyperparameters.

---

## 1. Setup & Workspace Preparation

### WHY?
Before writing any machine learning logic, we must establish a reproducible environment. This includes importing specialized tools for linear algebra, data structures, data visualization, and model estimation. Fixing the random state ensures that data splits are deterministic, allowing team members to replicate exact outputs.

### HOW?
We load standard libraries:
1. **NumPy** for fast matrix operations and vector mathematics.
2. **Pandas** to inspect and filter data tables.
3. **Matplotlib** & **Seaborn** to plot relationships.
4. **Scikit-learn** utilities for data splits and model validation metrics.

In [ ]:
# Set up mathematical and tabular workspaces
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Load standard datasets, linear classification estimators, and metrics
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Setting random seed guarantees that split indices remain identical across executions
np.random.seed(42)

# Enable plot rendering inline within the notebook interface
%matplotlib inline

## 2. Dataset Loading & Exploration

### WHY?
To build a model, we must first understand the structural layout of the dataset. Inspecting dimensions, data types, and statistics helps us identify missing entries, potential outliers, and target class balance before model fitting.

### HOW?
We load the **Iris Flower Dataset** (a benchmark 3-class target dataset containing 150 instances of flowers with measurements for sepal and petal length and width). We convert it into a Pandas DataFrame to inspect data types and statistics.

In [ ]:
# Load iris dataset into a structured pandas dataframe
iris = load_iris(as_frame=True)
df = iris.frame

# Output first five records to verify tabular structure
print('--- FIRST 5 ROWS ---')
print(df.head())

# Output column types and verify missing value counts
print('\n--- SCHEMAS & NULLS ---')
df.info()

# Print statistical characteristics (mean, variance, ranges)
print('\n--- STATISTICAL SUMMARY ---')
print(df.describe())

## 3. Data Preprocessing

### WHY?
Raw features must be cleaned and isolated before feeding them to optimization loops. Models expect separate inputs: a **Feature Matrix ($X$)** containing predictive features, and a **Target Vector ($y$)** containing the actual classes. Identifying missing entries and duplicates ensures that models do not learn from corrupted or repeated inputs.

### HOW?
We check the DataFrame for missing values and duplicate rows. We then separate the target label from the features, creating matrix $X$ and vector $y$.

In [ ]:
# Audit data quality by checking for null coordinates
missing_values = df.isnull().sum()
print('Null counts per column:\n', missing_values)

# Check for duplicate observations in the dataset
duplicates = df.duplicated().sum()
print('Duplicate rows:', duplicates)

# Isolate features (capital X matrix) from output classes (lowercase y vector)
X = df.drop('target', axis=1)
y = df['target']

# Verify target dimensions match
print('X Shape (Feature Matrix):', X.shape)
print('y Shape (Target Vector):', y.shape)

## 4. Mathematical Intuition: Computing Loss in NumPy

### WHY?
Machine Learning algorithms learn by optimizing parameters to minimize a loss function. Before using high-level libraries, writing a simple loss function manually in NumPy builds mathematical intuition for how error signals are calculated.

### HOW?
We define a simple mock linear problem: $y = 2x$. We define the Mean Squared Error (MSE) loss formula in Python, compute predictions on a mock input vector, and evaluate the loss.

In [ ]:
# Set up mock linear values to verify loss calculations
x_mock = np.array([1, 2, 3, 4])
y_true = np.array([2, 4, 6, 8])

# Define mock linear prediction function acting as our f(x)
def mock_predict(x):
    return 2 * x

# Compute Mean Squared Error: average of squared differences between actual and predicted
def compute_mse(actual, predicted):
    return np.mean((actual - predicted) ** 2)

# Evaluate predictions and compute final loss value
predictions = mock_predict(x_mock)
loss_val = compute_mse(y_true, predictions)

print('Predictions (f(x)):', predictions)
print('True Targets (y):', y_true)
print('Calculated Loss (MSE):', loss_val)

## 5. Model Training & Evaluation (Scikit-Learn)

### WHY?
To evaluate how well a model generalizes to new data, we must split our dataset into separate training and testing sets. Training the model on the training set minimizes empirical risk, while evaluating on the test set measures performance on unseen data, helping us diagnose potential overfitting.

### HOW?
We split our data ($80\%$ training, $20\%$ testing). We fit a **Logistic Regression** classifier on the training set, generate predictions on the test set, and evaluate the model using accuracy, a classification report, and a confusion matrix.

In [ ]:
# Split observations into separate training (80%) and testing (20%) datasets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Initialize and train Logistic Regression model on the training split
clf = LogisticRegression(max_iter=200)
clf.fit(X_train, y_train)

# Generate predictions on the unseen test split
y_pred = clf.predict(X_test)

# Evaluate model performance metrics
acc = accuracy_score(y_test, y_pred)
cm = confusion_matrix(y_test, y_pred)
report = classification_report(y_test, y_pred, target_names=iris.target_names)

print('Test Accuracy:', acc)
print('\nClassification Report:\n', report)
print('Confusion Matrix:\n', cm)

## 6. Data Visualization

### WHY?
Visualizations help us identify patterns and relationships within the features, such as feature correlation or cluster separation, showing how easily the target classes can be distinguished.

### HOW?
We use Seaborn and Matplotlib to plot target label distribution and visualize a scatter plot comparing sepal length and petal length.

In [ ]:
# Set up figure canvas for side-by-side plots
plt.figure(figsize=(12, 5))

# Plot target class distribution to audit balance
plt.subplot(1, 2, 1)
sns.countplot(x=y, hue=y, legend=False, palette='viridis')
plt.xticks(ticks=[0, 1, 2], labels=iris.target_names)
plt.title('Target Class Distribution')
plt.xlabel('Species')
plt.ylabel('Count')

# Visualize cluster separation comparing sepal vs. petal length
plt.subplot(1, 2, 2)
sns.scatterplot(data=df, x='sepal length (cm)', y='petal length (cm)', hue='target', palette='viridis')
plt.title('Sepal vs Petal Length Cluster Separation')
plt.xlabel('Sepal Length (cm)')
plt.ylabel('Petal Length (cm)')
plt.legend(title='Species', labels=iris.target_names.tolist())

plt.tight_layout()
plt.show()

## 7. Hyperparameter Experimentation

### WHY?
A model's performance depends heavily on its configuration parameters (hyperparameters), such as the size of the validation split or optimizer epoch limits. Iterating through different settings helps us find the optimal configuration for accuracy and generalization.

### HOW?
We loop through various train-test split sizes ($20\%$, $30\%$) and maximum iteration limits ($100$, $200$), training and evaluating a model for each combination.

In [ ]:
# Loop over configurations of test splits and training iteration epochs
for size in (0.2, 0.3):
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=size, random_state=42)
    for iterations in (100, 200):
        # Initialize model with current hyperparameter iteration settings
        clf_exp = LogisticRegression(max_iter=iterations, random_state=42)
        clf_exp.fit(X_tr, y_tr)
        preds = clf_exp.predict(X_te)
        accuracy = accuracy_score(y_te, preds)
        print(f'test_size={size:.1f}, max_iter={iterations}: accuracy = {accuracy:.4f}')

---

## 8. Summary & Quick Revision Guide

### Key Takeaways
*   **Traditional vs. ML**: Traditional programming processes rules and inputs to yield output. Machine Learning processes inputs and outputs to infer rules.
*   **Types of ML**: Supervised (regression/classification with labeled data), Unsupervised (clustering/dimensionality reduction with unlabeled data), and Reinforcement (agent learning from environmental rewards).
*   **ERM Framework**: We optimize training loss (using differentiable objectives like MSE or BCE) as a proxy for true risk.
*   **Differentiability**: Loss functions must be differentiable for optimization algorithms like Gradient Descent to calculate gradients and update model parameters.

### Common Mistakes
*   **Data Leakage**: Preprocessing the entire dataset (e.g., calculating scaling stats) before splitting into train and test sets.
*   **Evaluating with Accuracy alone**: Relying on accuracy for highly imbalanced datasets. Use F1-score, precision, or recall instead.
*   **Overfitting**: Training overly complex models on small datasets, causing them to memorize noise rather than generalize.

### Top Interview Questions
1.  **What is the difference between supervised and unsupervised learning?**  
    *Answer*: Supervised learning requires labeled targets. Unsupervised learning identifies patterns in unlabeled features.
2.  **Why can't raw accuracy be used as a loss function during model training?**  
    *Answer*: Accuracy is a non-differentiable step function. Its derivative is zero or undefined everywhere, providing no gradient signal to update parameters.
3.  **What is overfitting and how do you identify it?**  
    *Answer*: Overfitting occurs when a model memorizes training noise and fails to generalize. It is diagnosed by high training accuracy but low test accuracy.
4.  **What are common strategies to prevent overfitting?**  
    *Answer*: Apply regularization, collect more training data, simplify model architecture, use cross-validation, or apply early stopping.